# Exercicis de consolidació &mdash; Quant tardarà la meva comanda?

**MP 5134** &mdash; UT1 i UT2 &mdash; *Sessió de pràctica abans de l'examen*

*Dades: les comandes del 2025 de Picaporta, una app de repartiment de Manacor. Són
dades simulades, fetes perquè es comportin com les d'una empresa de debò.*

---
### Com funciona aquesta sessió

Avui no hi ha explicació: treballes tu. Aquest notebook s'assembla molt a l'examen
de la UT1 i la UT2. Tens un fitxer que no has vist mai i uns exercicis que barregen
codi i preguntes curtes, amb el pes posat a **entrenar models i avaluar-los**.

Unes quantes normes:

- Pots consultar tots els notebooks del curs i els solucionaris que tens a
  Classroom. **No pots fer servir cap eina d'intel·ligència artificial.** A l'examen
  tampoc no en tindràs, i avui és el dia de descobrir què saps fer sense ajuda.
- Tens **2 h 30 min com a màxim**. Cada part porta un temps orientatiu. Si
  t'encalles en un exercici, passa al següent i torna-hi després.
- Cada exercici té una cel·la de codi per a tu i, quan hi ha preguntes, una cel·la
  de **respostes**. Respon curt i amb números: *«el MAE baixa de 7,6 a 5,3
  minuts»* val més que un paràgraf. Si et cal més d'una cel·la de codi, afegeix-ne.
- La puntuació és orientativa, perquè puguis calcular com t'hauria anat si fos
  l'examen.

| Part | Què hi fas | Temps | Punts |
|---|---|---|---|
| A | Les dades: mirar-les, trobar-hi errors i netejar-les | 25 min | 2 |
| B | Partició, referències i una primera recta | 30 min | 2,5 |
| C | Millorar el model: més variables, la fuga i la corba de les hores | 45 min | 3,5 |
| D | El repte i el test | 40 min | 2 |

Al final hi ha dos exercicis extra, que no puntuen, per si et sobra temps.

## El cas: Picaporta

**Picaporta** és una app de Manacor que porta el menjar dels restaurants del centre
fins a casa teva: a Manacor mateix, a Son Macià, a Porto Cristo, a s'Illot i a Cala
Murada. Els repartidors van en bicicleta (només dins Manacor i fins a Son Macià) o
en moto.

Quan fas una comanda, l'app et diu quant tardarà. Ara mateix ho calcula amb una
regla que es van inventar els fundadors el primer dia:

> **temps promès = 30 minuts + 1,5 minuts per cada quilòmetre**

Els clients es queixen. De vegades el menjar arriba molt més tard del que deia
l'app, i d'altres el repartidor pica a la porta quan encara no han parat taula.
Picaporta t'ha contractat per fer un model que **predigui quants minuts tardarà una
comanda** a partir del que se sap en el moment de fer-la, i t'ha passat totes les
comandes del 2025.

> ⚠️ **En aquest fitxer hi ha una fuga d'informació.** Una de les columnes no es pot
> fer servir per predir, encara que el model hi surti guanyant. Trobar-la és una
> part essencial dels exercicis: un model que la fa servir dona uns números
> preciosos a l'ordinador i no serveix de res a l'empresa. Per trobar-la, fes-te
> aquesta pregunta per a cada columna: **quan el client fa la comanda, aquesta dada
> ja la sabem?**

### Les columnes

| Columna | Què és |
|---|---|
| `id_comanda` | número de la comanda |
| `data` | dia de la comanda |
| `hora` | hora de la comanda, en decimal: `13.5` són les 13:30 i `21.75`, les 21:45 |
| `cap_de_setmana` | 1 si és dissabte o diumenge, 0 si no |
| `zona` | on s'entrega: Manacor, Son Macià, Porto Cristo, s'Illot o Cala Murada |
| `distancia_km` | quilòmetres del restaurant fins a casa del client |
| `plats` | nombre de plats de la comanda |
| `temps_cuina_min` | minuts que el restaurant diu que tardarà a tenir la comanda a punt |
| `moto` | 1 si el repartidor va en moto, 0 si va en bicicleta |
| `experiencia_mesos` | mesos que fa que el repartidor treballa a Picaporta |
| `pluja` | 1 si plovia quan es va fer la comanda, 0 si no |
| `temperatura` | temperatura a Manacor en aquell moment (°C) |
| `valoracio_client` | estrelles, d'1 a 5, que el client ha posat a la comanda a l'app |
| `minuts_entrega` | minuts des que el client fa la comanda fins que li arriba a casa |

La cel·la següent carrega el fitxer. Ja està feta: només l'has d'executar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT02-Regressio_lineal_polinomica/consolidacio_picaporta/repartiments_manacor.csv"
df = pd.read_csv(URL_DADES, parse_dates=["data"])

df.head()

## Part A · Les dades
*25 minuts · 2 punts*

### Exercici 1 · Primer cop d'ull *(0,75 punts)*

Fes-te una idea del fitxer amb les eines de sempre (`shape`, `info`, `describe`,
`value_counts`...) i respon:

1. Quantes comandes i quantes columnes té el fitxer?
2. Quina és la variable objectiu? És un problema de regressió o de classificació?
   Per què?
3. Quines columnes **no** faries servir com a entrada del model? Per a cadascuna,
   digues per què en una frase. La fuga d'informació ha de sortir aquí.

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 
3. 

### Exercici 2 · Errors, buits i valors estranys *(1,25 punts)*

Abans d'entrenar res has de saber si et pots fiar de les dades. Busca:

- **Valors absents**: a quines columnes n'hi ha, i quants.
- **Valors impossibles**: dades que no poden ser certes. Un histograma de les
  columnes numèriques, o un `describe()` mirat amb calma, t'hi ajudaran. Si un
  histograma surt tot aixafat en una sola barra, demana't per què.

Perquè tota la classe treballi amb les mateixes dades i els resultats del repte es
puguin comparar, aquí la decisió ja està presa: **elimina totes les files que tinguin
algun buit o algun valor impossible**, i deixa les dades bones en un DataFrame nou que
es digui `net`. La teva feina és trobar-les i explicar, per a cada problema, per què
aquelles files no ens serveixen.

Respon també:

- Si no les haguessis d'eliminar, què podries fer amb les comandes a les quals falta
  el `temps_cuina_min`?
- Entre les comandes que semblen correctes, n'hi ha unes quantes que van tardar més
  d'una hora. Les treus? Per què?

> **Comprovació.** Després de netejar, `net` ha de tenir **2.895 comandes**. Si en
> tens unes altres, revisa la neteja abans de continuar: totes les xifres del repte
> depenen d'aquest número.

In [ ]:
# Escriu aquí el teu codi

**Respostes**

- Valors absents, i per què no ens serveixen:
- Valors impossibles, i per què no ens serveixen:
- Què podria fer amb els buits de `temps_cuina_min`:
- Les comandes de més d'una hora:

## Part B · Partició, referències i una primera recta
*30 minuts · 2,5 punts*

### Exercici 3 · Entrenament, validació i test *(0,5 punts)*

Parteix `net` en tres trossos amb `train_test_split`, igual que al NB 2.2:

- primer aparta un 20% de les comandes per al **test** (`test`);
- després parteix la resta en **entrenament** (`train`, el 75%) i **validació**
  (`valid`, el 25%).

Fes servir `random_state=42` a les dues particions i mostra quantes comandes té cada
tros. Des d'ara, **el test queda segellat**: no el miris ni hi calculis res fins a
l'exercici 10.

Respon:

1. Per què és important que tota la classe faci servir el mateix `random_state`?
2. Per què el test s'aparta ara, abans de provar cap model?

> **Comprovació.** Et queden 1.737 comandes d'entrenament, 579 de validació i 579 de
> test.

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 

### Exercici 4 · Les referències *(1 punt)*

Un número tot sol no vol dir res. Abans de cap model has de saber quin número cal
batre. Calcula el **MAE, el RMSE i el R2 sobre la validació** de dues referències:

1. un `DummyRegressor` que digui sempre la mitjana de l'entrenament;
2. la regla que fa servir ara Picaporta: 30 minuts més 1,5 minuts per km.

Posa els resultats en una taula, amb una fila per model. Et convé escriure una
funció `mesurar(y_real, y_predit)` que torni les tres mètriques, perquè la faràs
servir fins al final.

Respon:

1. La regla de Picaporta és millor que dir sempre la mitjana? De quants minuts?
2. El R2 del `DummyRegressor` surt pràcticament 0. Per què?
3. Si haguessis d'explicar a la gerent de Picaporta quant s'equivoca la regla, quin
   número li donaries, i com li ho diries en una frase?

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 
3. 

### Exercici 5 · Una recta amb la distància *(1 punt)*

Entrena una regressió lineal que predigui `minuts_entrega` només a partir de
`distancia_km`.

1. Segons el model, quants minuts afegeix cada quilòmetre? I quina és l'ordenada a
   l'origen? Compara-ho amb la regla de Picaporta.
2. Afegeix les mètriques de validació de la recta a la taula de l'exercici 4. La
   recta guanya la regla?
3. Dibuixa el núvol de punts de la distància contra els minuts, **pintant cada
   comanda d'un color segons la columna `moto`** (amb el paràmetre `c=` de
   `scatter`), i la recta per damunt. Et sembla que les bicicletes i les motos
   segueixen la mateixa recta? Guarda't la resposta: et farà falta al repte.

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 
3. 

## Part C · Millorar el model
*45 minuts · 3,5 punts*

### Exercici 6 · Totes les variables *(1 punt)*

Entrena una regressió lineal amb **totes les columnes que a l'exercici 1 has decidit
que es poden fer servir**.

1. Calcula el MAE d'entrenament i el de validació. Hi ha sobreajust? Com ho saps?
2. Mira els coeficients. Quants minuts afegeix la pluja? I quants en treu anar en
   moto en lloc d'anar en bicicleta?
3. Hi ha alguna variable que no aporti res? Treu-la, torna a entrenar i comprova si
   el MAE de validació canvia.

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 
3. 

### Exercici 7 · Caçar la fuga *(1 punt)*

Ara comprovaràs si a l'exercici 1 vas encertar la fuga. Afegeix al model de
l'exercici 6 **la columna que creus que és la fuga** i torna a mesurar la validació.

1. Quant baixa el MAE? I quant puja el R2?
2. Explica per què aquesta columna fa «millorar» tant el model i per què Picaporta no
   la podria fer servir mai per predir una comanda nova.

Si a l'exercici 1 no la vas trobar, prova d'afegir d'una en una les columnes
numèriques que hi vas deixar fora. La que fa baixar el MAE d'una manera
espectacular és la fuga.

**A partir d'aquí, la fuga queda fora de tots els models.**

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 

### Exercici 8 · La corba de les hores *(1,5 punts)*

1. Dibuixa l'hora de la comanda contra els minuts d'entrega i, per damunt, **la
   mitjana de minuts de cada hora** (amb `groupby`, com la mitjana per franja de
   latitud del NB 2.2). Quina forma té? Explica-la pensant en com funciona un
   restaurant. Una regressió lineal la pot seguir?
2. Al model de l'exercici 6, canvia la columna `hora` per la **corba de l'hora**: les
   potències de l'hora fabricades amb `PolynomialFeatures`, com vam fer amb la
   temperatura a la secció 8 del NB 2.3. Prova els graus de l'1 al 8 i fes una taula
   amb el MAE d'entrenament i el de validació de cada grau.
3. Quin grau tries, i per què? Per què els graus 2 i 3 gairebé no milloren res? I
   per què el grau s'ha de triar mirant la validació i no l'entrenament?

*Pistes:* escala l'hora abans de fabricar-ne les potències (per exemple, `hora / 24`),
com vam dividir la temperatura entre 30. La fàbrica es prepara amb l'entrenament
(`fit_transform`) i a la validació només s'aplica (`transform`). Per ajuntar les
columnes que surten de la fàbrica amb les altres, fes servir `np.hstack`, també com
a la secció 8 del NB 2.3.

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 
3. 

## Part D · El repte i el test
*40 minuts · 2 punts*

### Exercici 9 · El repte de Picaporta *(1,25 punts)*

La gerent de Picaporta t'ha dit quin error està disposada a acceptar. Com més avall
el deixis, més contents estaran els clients:

| Nivell | MAE de validació | Punts |
|---|---|---|
| 🥉 Bronze | per sota de **6,0 minuts** | 0,5 |
| 🥈 Plata | per sota de **4,5 minuts** | 1 |
| 🥇 Or | per sota de **3,8 minuts** | 1,25 |

Les regles del joc:

- Entrenes només amb `train` i compares amb `valid`. El test no es toca.
- La columna de la fuga no hi pot ser.
- No pots treure comandes reals de la validació perquè l'error baixi: la validació
  és la de l'exercici 3.
- Pots fer servir tot el que hem vist: regressió lineal, `PolynomialFeatures`,
  columnes fabricades a mà, treure'n alguna...

Fes una taula amb **tots els intents** (com a mínim tres), amb el nombre de columnes,
el MAE d'entrenament, el MAE de validació i el R2 de validació de cadascun, i marca
quin és el teu model final.

Respon:

1. Quin nivell has aconseguit, i amb quin model?
2. En algun intent, l'error d'entrenament ha baixat i el de validació no ho ha fet, o
   fins i tot ha pujat? Què vol dir? Si no t'ha passat, prova `PolynomialFeatures` de
   grau 3 o 4 sobre totes les variables alhora i mira què passa.

<details>
<summary><b>Pistes</b> (obre-les només si les necessites)</summary>

- La plata surt de l'exercici 8.
- Per a l'or, torna al gràfic de l'exercici 5. Si les bicicletes i les motos no fan
  la mateixa recta, el model ha de poder fer servir un pendent diferent per a cada
  vehicle. Hi ha dues maneres de donar-li: fabricar a mà una columna que ho expressi,
  o deixar que `PolynomialFeatures` multipliqui les variables entre elles.

</details>

**Rànquing de la classe:** quan acabis, digues al professor el teu millor MAE de
validació. Qui el deixarà més avall?

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 

### Exercici 10 · Obrir el test *(0,75 punts)*

Ara, i només ara, obre el test. Fes-ho una sola vegada i amb el model final del
repte tal com l'has entrenat, sense tocar-hi res.

1. Calcula el MAE, el RMSE i el R2 sobre el test. S'assemblen als de validació?
2. Per què el test s'ha de mirar una sola vegada i al final? Què passaria si ara
   canviessis el model perquè el test sortís millor?
3. Escriu el missatge que enviaries a la gerent de Picaporta, en dues o tres frases:
   quant s'equivoca de mitjana el model nou i com queda comparat amb la regla que fan
   servir ara. Per comparar-ho bé, calcula també el MAE de la regla sobre el test.

In [ ]:
# Escriu aquí el teu codi

**Respostes**

1. 
2. 
3. 

## Exercicis extra
*Si t'ha sobrat temps. No puntuen.*

### Exercici 11 · Amb molt poques dades

Imagina que t'haguessin encarregat el model la primera setmana, quan Picaporta només
tenia 60 comandes. Per simular-ho, fes servir només les 60 primeres files de
l'entrenament (`train.head(60)`). Entrena amb aquestes 60 comandes la regressió
lineal amb totes les variables i, després, la mateixa amb `PolynomialFeatures` de
grau 2 i de grau 3 sobre totes les variables. Mesura cada model amb l'entrenament i
amb la validació.

Què passa? Quantes columnes té cada model, comparat amb les 60 comandes que té per
aprendre?

In [ ]:
# Escriu aquí el teu codi

**Resposta:**

### Exercici 12 · Els pitjors errors

Amb el teu model final, fes una taula amb el valor real, el predit i l'error de cada
comanda de validació, ordenada de l'error més gran al més petit. Les deu comandes on
més s'equivoca, tenen alguna cosa en comú? Són errors de les dades o comandes reals
difícils? Calcula també el quocient entre el RMSE i el MAE: què et diu?

In [ ]:
# Escriu aquí el teu codi

**Resposta:**

---
### Quan acabis

Desa el notebook al teu Drive. Quan es pengi el solucionari a Classroom, compara-hi
les teves respostes, sobretot les dels exercicis on has dubtat: és allà on hi ha la
feina de cara a l'examen.